# Cvxpy и Scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import warnings
warnings.simplefilter("ignore")

import cvxpy as cp
import scipy

__Задача 1.__ Задача о многопродуктовом потоке возникает, когда потоки между разными парами источников-стоков делят пропускные способности дуг общей сети, как, например, в случае интернет-трафика, автомобильного трафика, перемещения грузов.

Пусть дан двунаправленный граф с множеством городов $\mathcal{V} = \{1, \ldots, n \}$ и множеством односторонних дорог $\mathcal{E} = \{1, \ldots, m \}$. Для каждой дороги $e \in \mathcal{E}$ определим стоимость $c_e$ (costs) перемещенения единицы груза и пропускную способность $b_e$ (capacities). Пусть для каждой пары вершин $(i, j) \in \mathcal{V} \times \mathcal{V}$ задано $d_{ij}$ — количество груза, необходимое переместить из города $i$ в город $j$ (traffic matrix). Тогда задачу о минимальной по стоимости транспортировке грузов можно сформулировать так:

$$\begin{align*}
\min_{f^{ij}_e \in \mathbb{R}} ~~&~~ \sum_{e \in \mathcal{E}} c_e \sum_{i,j \in \mathcal{V}} f^{ij}_e \\ \text{s.t.} ~~&~~  f_e^{ij} \geq 0, \quad \forall e \in \mathcal{E},~ \forall i,j \in \mathcal{V} \\ 
~~&~~ \sum_{i, j \in \mathcal{V}}f^{ij}_e \leq b_e, \quad \forall e \in \mathcal{E} \\ 
~~&~~ \sum_{e \in \mathcal{E}} A_{ve} f^{ij}_e = \begin{cases} -d_{ij}, & v = i \\ d_{ij}, & v = j \\ 0, & иначе \end{cases} \quad \forall v, i, j \in \mathcal{V},
\end{align*}$$

где $A$ — матрица инцидентности графа (incidence_matrix).

In [ ]:
incidence_matrix = np.array([[-1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,-1,-1,-1,-1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0],[0,0,0,0,0,-1,-1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,-1,-1,-1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0],[0,0,1,0,0,0,0,0,0,0,-1,-1,-1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0],[0,0,0,1,0,1,0,0,0,0,0,0,0,-1,-1,-1,0,0,1,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,-1,-1,-1,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,-1,-1,0,0,0,1,0,0,0,0,0],[0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,-1,0,0,0,0,0,0,1],[0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,-1,-1,-1,0,1,0,0],[0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,-1,-1,0,0],[0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,-1,-1]])
traffic_matrix = np.array([[0.,0.3,0.7,0.1,0.4,0.2,0.1,0.2,0.3,0.1,0.1,1.3],[0.5,0.,1.4,0.8,13.2,2.1,0.6,16.2,2.1,0.5,0.6,8.7],[0.7,8.6,0.,7.4,77.6,3.4,7.9,90.8,5.6,0.8,1.2,5.2],[0.4,0.8,3.3,0.,1.8,1.8,1.,1.8,1.5,5.2,1.5,2.4],[0.2,6.1,3.,0.3,0.,0.8,0.5,3.9,1.6,0.1,0.4,3.6],[0.5,1.9,6.4,2.5,3.5,0.,1.1,5.3,4.4,0.5,0.8,3.1],[0.2,0.5,1.8,0.9,0.7,0.9,0.,0.7,0.9,0.3,0.5,1.1],[0.3,10.5,100.,1.5,38.,2.8,0.7,0.,2.8,5.4,2.3,16.8],[0.3,4.2,28.8,2.7,7.8,2.1,1.6,8.,0.,1.2,2.,11.3],[0.2,0.2,1.6,1.2,0.7,0.8,0.5,2.3,1.3,0.,1.4,0.9],[0.2,9.7,5.7,1.8,1.8,7.,3.2,8.6,3.,6.2,0.,3.8],[0.5,6.,8.1,3.8,6.3,2.4,2.4,10.4,9.4,1.5,1.9,0.]])
capacities = 200 * np.array([1,1,1,0.25,1,1,1,1,1,1,1,1,1,0.25,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1])
nodes, edges = incidence_matrix.shape
costs = np.ones(edges)

__а) (0.5 балла)__ Объясните, какой физический смысл имеют переменные $f_{e}^{ij}$.

In [ ]:
# Ваше решение (Markdown)

__б) (1.5 балла)__ С помощью `cvxpy` решите задачу. Выведите ответ.

In [ ]:
# Ваше решение (Code)

Постройте график оптимального суммарного потока по дороге от индекса дороги.

In [ ]:
# Ваше решение (Code)

__в) (1.5 балла)__ Будем считать, что в качестве решения нас интересует только суммарный поток на каждой дороге. Используюйте переменные, описывающие входящие и исходящие потоки. Переформулируйте задачу, уменьшив число переменных.

In [ ]:
# Ваше решение (Markdown)

Решите задачу в новой постановке с помощью `cvxpy`. Выведите ответ.

In [ ]:
# Ваше решение (Code)

Постройте график оптимального суммарного потока по дороге от индекса дороги.

In [ ]:
# Ваше решение (Code)

__г) (1.5 балла)__ Теперь можно увеличить пропускные способности дорог на произвольные величины, но суммарно не превышающие $b_{add}$. Также хочется учесть, что на любой дороге может произойти авария, что выразится в уменьшении её пропускной способности в два раза. Адаптируйте задачу под такую постановку.

In [ ]:
# Ваше решение (Markdown)

Возьмите `b_add = 10` и решите задачу в новой постановке с помощью `cvxpy` в худшем случае, то есть по всем возможным авариям. Выведите ответ.

In [ ]:
# Ваше решение (Code)

Постройте график оптимального суммарного потока по дороге от индекса дороги.

In [ ]:
# Ваше решение (Code)

__Задача 2.__ Рассмотрим задачу максимального разреза графа. Дан связный ненаправленный граф $G = (V, E)$ с $N$ вершинами, причем между некоторыми имеется ребро заданного веса $w_{ij}$ $(w_{ii} = 0)$. Необходимо разделить вершины на две группы так, чтобы максимизировать сумму весов всех ребер, которые соединяют вершины из разных групп. Наша задача может быть записана как задача целочисленного квадратичного программирования:

$$\begin{align*}
\max_{x \in \mathbb{R}^N} ~~&~~ \frac{1}{2} \sum_{i = 1}^{N} \sum_{j = 1}^{N} w_{ij} \frac{(1 - x_i x_j)}{2} \\
\text{s.t.} ~~&~~ x_i \in \{1, -1\}, \quad \forall i \in V.
\end{align*}$$

$\frac{(1-x_i x_j)}{2}$ равно единице тогда и только тогда, когда обе вершины находятся в разных группах, в противном случае 0. Поэтому мы максимизируем как раз сумму весов тех ребер, что соединяют вершины из разных групп. Деление на два происходит из-за того, что каждое ребро считается два раза.

Основная проблема заключается в том, что без модификаций это NP-трудная задача, для которой неизвестен алгоритм решения за полиномиальное время. Для их решения обычно проводится релаксация условия таким образом, что бы получилась новая задача, для которой известен алгоритм решения за полиномиальное время, решение которой близко к настоящему решению.

Мы сделаем релаксацию к задаче полуопределенного программирования, которая в общем виде задается следующим образом:

$$ \begin{align*}
\min_{X \in \mathbb{R}^{N \times N}} ~~&~~ \langle C, X \rangle \\
\text{s.t.} ~~&~~ \langle A_i, X \rangle \le b_i, \quad \forall i \in V \\
~~&~~ X \succeq 0.
\end{align*}
$$

__а) (0.5 балла)__ Перейдите от чисел $x_i$, равных по модулю единице, к единичным векторам $u_i \in \mathbb{R}^N$. Запишите исходную задачу с новыми условиями и переменными. Переформулируйте задачу.

In [ ]:
# Ваше решение (Markdown)

__б) (0.5 балла)__ Теперь сделаем из квадратичной функции линейную. Введите $Y_{i, j} = u_i^\top u_j$ и перепишите задачу из предыдущего пункта с новыми условиями. Сделайте так, чтобы условия выглядели так же, как в шаблоне для задачи полуопределенного программирования.

In [ ]:
# Ваше решение (Markdown)

__в) (0.5 балла)__ Представьте целевую функцию в виде произведения матриц, как в шаблоне.

In [ ]:
# Ваше решение (Markdown)

__г) (0.5 балла)__ Ниже задана матрица весов $W$. Вам нужно решить для $Y$ задачу, используя `cvxpy`.

In [ ]:
W = np.array(
    [
        [0, 4, 10, -3, 0, 0, 0],
        [4, 0, 0, 0, 6, 0, 0],
        [10, 0, 0, 7, 0, 0, 0],
        [-3, 0, 7, 0, 0, 0, 8],
        [0, 6, 0, 0, 0, 4, 0],
        [0, 0, 0, 0, 4, 0, 1],
        [0, 0, 0, 8, 0, 1, 0],
    ],
    dtype=np.float32,
)
num_of_nodes = 7

In [ ]:
# Ваше решение (Code)

In [ ]:
assert abs(optimal_value - 39) < 1e-2, "Неправильно"

__д) (0.5 балла)__ Мы получили решение для $Y$. Однако мы хотим узнать значения $x_i$. Начнем идти в обратном порядке. Используя наше определение матрицы $Y$, запишите образующую $Y$ матрицу.

In [ ]:
# Ваше решение (Code)

__е) (0.5 балла)__ Мы получили матрицу $U$, у которой $i$-ая строка равна $u_i$ единичному вектору. Он олицетворяет принадлежность точки тому или иному набору. Нам нужно этот вектор превратить в цифру 1 или -1. Наша модель обучена так, что вершины из разных наборов имеют "разные" векторные репрезентации, которые находятся в разных частях единичной сферы. Мы воспользуемся следующим приемом (_randomized hyperplane rounding_): нормально сгенерируем случайную плоскость и поделим точки на единичной сфере на 2 набора в зависимости от того, по какую сторону эти точки оказались относительно сгенерированной плоскости. Мы полагаем, что если две точки находятся в разных частях сферы, то, скорее всего, при создании случайной плоскости через центр сферы эти точки окажутся по разные стороны.

In [ ]:
S = []
S_bar = []

# YOUR CODE HERE

In [ ]:
def calculate_cut(S, S_bar, W):
    max_cut = 0
    for i in S:
        for j in S_bar:
            max_cut += W[i][j]
    return max_cut

assert (abs(calculate_cut(S, S_bar, W) - 39) < 1e-2), "Неправильно"

__Задача 3.__ Пусть самолет движется по заданному маршруту, состоящему из $n$ участков, соединяющих $n + 1$ путевую точку с индексами от $0$ до $n$. Участок $i$ начинается на путевой точке $i - 1$ и заканчивается на путевой точке $i$. Самолет стартует в момент времени $t = 0$ с путевой точки $0$. Он движется по участку $i$ с постоянной скоростью $s_i$. Для скоростей на каждом участке заданы нижние и верхние границы: $s_i^{\min} \leq s_i \leq s_i^{\max}$. Самолет не останавливается на путевых точках, он сразу же продолжает движение к следующему участку. Длина участка $i$ равна $d_i > 0$, соотвественно время прохождения этого участка равно $\frac{d_i}{s_i}$. Пусть $\tau_i$ обозначает время прибытия самолета на путевую точку $i$. Самолет должен прибыть на путевую точку $i$  между заранее заданными моментами времени: $\tau_i^{\min} \leq \tau_i \leq \tau_i^{\max}$. Скорость расхода топлива (кг/c) зависит от скорости движения и выражается выпуклой возрастающей функцией

$$
\Phi(s_i) = a s_i^2 + b s_i + c,
$$

где значения параметров равны $a = 1$, $b = 6$, $c = 10$.

Ваша задача — выбрать скорости $\{s_i\}_{i = 1}^n$ так, чтобы минимизировать суммарный расход топлива.

In [ ]:
d = np.array([1.9501,1.2311,1.6068,1.486,1.8913,1.7621,1.4565,1.0185,1.8214,1.4447,1.6154,1.7919,1.9218,1.7382,1.1763,1.4057,1.9355,1.9169,1.4103,1.8936,1.0579,1.3529,1.8132,1.0099,1.1389,1.2028,1.1987,1.6038,1.2722,1.1988,1.0153,1.7468,1.4451,1.9318,1.466,1.4186,1.8462,1.5252,1.2026,1.6721,1.8381,1.0196,1.6813,1.3795,1.8318,1.5028,1.7095,1.4289,1.3046,1.1897,1.1934,1.6822,1.3028,1.5417,1.1509,1.6979,1.3784,1.86,1.8537,1.5936,1.4966,1.8998,1.8216,1.6449,1.818,1.6602,1.342,1.2897,1.3412,1.5341,1.7271,1.3093,1.8385,1.5681,1.3704,1.7027,1.5466,1.4449,1.6946,1.6213,1.7948,1.9568,1.5226,1.8801,1.173,1.9797,1.2714,1.2523,1.8757,1.7373,1.1365,1.0118,1.8939,1.1991,1.2987,1.6614,1.2844,1.4692,1.0648,1.9883])
smin = np.array([0.7828,0.6235,0.7155,0.534,0.6329,0.4259,0.7798,0.9604,0.7298,0.8405,0.4091,0.5798,0.9833,0.8808,0.6611,0.7678,0.9942,0.2592,0.8029,0.2503,0.6154,0.505,1.0744,0.215,0.968,1.1708,1.1901,0.9889,0.6387,0.6983,0.414,0.8435,0.52,1.1601,0.9266,0.612,0.9446,0.4679,0.6399,1.1334,0.8833,0.4126,1.0392,0.8288,0.3338,0.4071,0.8072,0.8299,0.5705,0.7751,0.6514,0.2439,0.2272,0.5127,0.2129,0.584,0.8831,0.2928,0.2353,0.8124,0.8085,0.2158,0.2164,0.3901,0.7869,0.2576,0.5676,0.8315,0.9176,0.8927,0.2841,0.6544,0.6418,0.5533,0.3536,0.8756,0.8992,0.9275,0.6784,0.7548,0.321,0.6508,0.9159,1.0928,0.4731,0.4548,1.0656,0.4324,1.0049,1.1084,0.4319,0.4393,0.2498,0.2784,0.8408,0.3909,1.0439,0.3739,0.3708,1.1943])
smax = np.array([1.9624,1.6036,1.6439,1.5641,1.7194,1.909,1.3193,1.3366,1.947,2.8803,2.5775,1.4087,1.6039,2.9266,1.4369,2.3595,3.228,1.889,2.8436,0.5701,1.1894,2.4425,2.2347,2.2957,2.7378,2.8455,2.1823,1.6209,1.2499,1.3805,1.5589,2.8554,1.8005,3.092,2.1482,1.8267,2.1459,1.5924,2.7431,1.4445,1.7781,0.8109,2.7256,2.429,2.5997,1.8125,1.9073,1.5275,2.1209,2.5419,1.7032,0.5636,1.3669,2.32,2.1006,2.7239,2.8726,1.3283,1.7769,2.575,1.4963,2.3254,1.6548,1.9537,1.5557,1.6551,2.7307,1.8018,2.5287,1.9765,1.8387,2.3525,1.7362,1.6805,1.964,2.8508,1.9424,2.078,2.1677,2.1863,2.0541,1.9734,2.7687,2.3715,1.1449,2.156,3.331,2.3456,2.712,2.3783,0.9611,2.069,1.2805,0.8585,2.2744,2.3369,2.6918,2.6728,2.5941,1.612])
tau_min = np.array([1.0809,2.7265,3.5118,5.3038,5.4516,7.1648,9.2674,12.1543,14.4058,16.6258,17.9214,19.8242,22.2333,22.4849,25.3213,28.0691,29.8751,30.6358,33.2561,34.7963,36.9943,38.261,41.1451,41.3613,43.0215,43.8974,46.4713,47.4786,49.5192,49.6795,50.7495,52.2444,53.5477,55.2351,57.085,57.425,60.1198,62.3834,64.7568,67.2016,69.2116,69.8143,70.6335,72.5122,74.1228,74.3013,74.5682,75.3821,76.6093,78.0315,80.7584,82.5472,83.534,84.9686,86.7601,87.2445,89.7329,92.6013,94.3879,94.4742,96.9105,98.7409,100.8453,101.1219,102.3966,103.5233,104.0218,106.5212,109.0372,110.392,113.2618,113.7033,116.3131,118.6214,119.9539,121.8157,124.6708,126.5908,127.3328,128.3909,128.9545,130.4264,131.6542,133.0448,134.8776,135.0912,136.034,137.8591,138.3842,140.2473,140.9852,142.7472,144.2654,145.6597,147.284,150.111,151.1363,152.3417,153.2647,154.4994])
tau_max = np.array([4.6528,6.5147,7.5178,9.7478,9.0641,10.3891,13.154,16.0878,17.4352,20.9539,22.3695,23.3875,25.7569,26.9019,29.889,33.0415,33.8218,35.4414,37.1583,39.4054,41.652,41.5935,44.9329,45.4028,47.4577,48.0358,50.3929,51.3692,52.6947,53.5665,54.4821,55.8495,58.2514,59.7541,61.9845,61.5409,63.1482,66.5758,69.3892,72.1558,72.6555,74.2216,74.6777,77.378,78.5495,77.7574,78.4675,78.7265,81.547,81.7429,83.8565,87.0579,88.3237,88.5409,90.2625,92.11,92.9949,97.4829,98.7916,99.1695,100.3291,102.651,104.0075,105.8242,106.5207,107.1619,107.7716,111.2568,112.7815,113.5394,116.6615,116.8022,120.4465,121.8652,123.9981,125.0498,129.2106,130.3409,131.9796,131.4842,133.1503,135.3247,135.2318,137.8225,138.0808,138.2218,139.5026,142.7253,141.5105,143.7757,145.9842,146.1712,148.2622,149.2407,151.6295,155.027,155.6694,156.6739,156.5266,157.6903])

__а) (1 балл)__ Сформулируйте задачу в виде задачи выпуклой оптимизации. 

In [ ]:
# Ваше решение (Markdown)

__б) (1 балл)__ Решите задачу с помощью `cvxpy`. Выведите значение оптимального расхода топлива.

In [ ]:
# Ваше решение (Code)